
# Reel Insights: SQL Analysis

This notebook uses SQL to analyse the global movie industry dataset.

We will use SQLite to answer questions about movie revenue, genres, actors, directors, and keywords.


In [2]:

import pandas as pd
import sqlite3
import os

print("Tools ready!")

Tools ready!


In [3]:
database_path = "../data/reel_insights.db"

connection = sqlite3.connect(database_path)

print("Database connected successfully!")

Database connected successfully!


In [58]:

files = {
    "movies": "movies_cleaned_corrected.csv",
    "genres": "genres_cleaned.csv",
    "movie_genres": "movie_genres_cleaned.csv",
    "cast": "cast_cleaned.csv",
    "crew": "crew_cleaned.csv",
    "movie_keywords": "movie_keywords_cleaned.csv"
}
for table_name, file_name in files.items():
    file_path = "../data/" + file_name
    data = pd.read_csv(file_path)
    data.to_sql(table_name, connection, if_exists="replace", index=False)
    print(table_name, "table created:", len(data), "rows")

print("All six tables are ready!")


movies table created: 2503 rows


genres table created: 19 rows
movie_genres table created: 6878 rows
cast table created: 24902 rows
crew table created: 7289 rows
movie_keywords table created: 41681 rows
All six tables are ready!


In [4]:
# Check the tables stored in the SQLite database

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    ORDER BY name;
    """,
    connection
)

print("Tables in the Reel Insights database:")
print(tables)

Tables in the Reel Insights database:
             name
0            cast
1            crew
2          genres
3    movie_genres
4  movie_keywords
5          movies


In [59]:

query = """
SELECT title, release_date, vote_average
FROM movies
LIMIT 5;
"""

result = pd.read_sql_query(query, connection)

print(result.to_string(index=False))


          title release_date  vote_average
   Interstellar   2014-11-05         8.484
      Inception   2010-07-15         8.400
   The Avengers   2012-04-25         8.062
The Dark Knight   2008-07-16         8.534
         Avatar   2009-12-16         7.610


In [60]:
check = pd.read_sql_query("""
    SELECT
        COUNT(*) AS movie_count,
        SUM(CASE WHEN budget = 0 THEN 1 ELSE 0 END)
            AS zero_budgets,
        SUM(CASE WHEN revenue = 0 THEN 1 ELSE 0 END)
            AS zero_revenues,
        SUM(CASE WHEN budget IS NULL THEN 1 ELSE 0 END)
            AS missing_budgets,
        SUM(CASE WHEN revenue IS NULL THEN 1 ELSE 0 END)
            AS missing_revenues
    FROM movies;
""", connection)

print(check.to_string(index=False))

 movie_count  zero_budgets  zero_revenues  missing_budgets  missing_revenues
        2503            90            102                0                 0


In [61]:

query1 = """
SELECT m.title, m.release_date
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action'
  AND m.release_date > '2015-12-31'
ORDER BY m.release_date;
"""

result1 = pd.read_sql_query(query1, connection)

print("Action movies released after 2015:")
print(result1.to_string(index=False))
print("\nTotal movies found:", len(result1))


Action movies released after 2015:
                                                               title release_date
                                                        The 5th Wave   2016-01-14
                                                     Kung Fu Panda 3   2016-01-23
                                                         Zoolander 2   2016-02-06
                                                            Deadpool   2016-02-09
                                                            Triple 9   2016-02-19
                                                             Grimsby   2016-02-24
                                                       Gods of Egypt   2016-02-25
                                                   They Call Me Jeeg   2016-02-25
                                                   London Has Fallen   2016-03-02
                                                           Allegiant   2016-03-09
                                  Batman v Superman: Dawn of Ju

**Insight**: This query identifies Action movies released after 2015, showing how the dataset can be filtered by both genre and release date.

In [63]:

query2 = """
SELECT
    m.title,
    GROUP_CONCAT(DISTINCT g.genre_name) AS genres,
    m.revenue
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
WHERE m.revenue > 0
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10;
"""

result2 = pd.read_sql_query(query2, connection)

print("Top 10 highest-grossing movies:")
print(result2.to_string(index=False))


Top 10 highest-grossing movies:
                       title                                    genres      revenue
                      Avatar          Adventure,Action,Science Fiction 2923706026.0
           Avengers: Endgame          Adventure,Action,Science Fiction 2799439100.0
    Avatar: The Way of Water          Adventure,Action,Science Fiction 2334484620.0
                     Titanic                             Drama,Romance 2264162353.0
Star Wars: The Force Awakens          Adventure,Action,Science Fiction 2068223624.0
      Avengers: Infinity War          Adventure,Action,Science Fiction 2052415039.0
     Spider-Man: No Way Home          Adventure,Action,Science Fiction 1921206586.0
                  Zootopia 2 Adventure,Animation,Comedy,Mystery,Family 1868208796.0
                Inside Out 2         Adventure,Animation,Comedy,Family 1698863816.0
              Jurassic World        Adventure,Thriller,Science Fiction 1671537444.0


**Insight**: The highest-grossing movies in the dataset can be compared alongside their genres. This helps show which films generated the greatest recorded revenue.

In [65]:

query3 = """
SELECT
    g.genre_name,
    ROUND(AVG(NULLIF(m.budget, 0)), 2) AS average_budget,
    ROUND(AVG(NULLIF(m.revenue, 0)), 2) AS average_revenue
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
ORDER BY average_revenue DESC;
"""

result3 = pd.read_sql_query(query3, connection)

print("Average budget and revenue by genre:")
print(result3.to_string(index=False))

Average budget and revenue by genre:
     genre_name  average_budget  average_revenue
      Adventure    110673597.60     400311337.47
      Animation     87635507.97     371835705.40
         Family     87906297.35     353369109.61
Science Fiction     96722245.91     333648843.96
         Action     94227512.51     308453177.92
        Fantasy     88640066.03     302705993.38
         Comedy     54500177.39     221526373.87
          Music     42817021.28     201566173.27
        Romance     36939702.75     174811036.03
       Thriller     51214051.11     168004244.88
            War     55271158.54     166091464.61
          Crime     46608459.46     151931527.92
        Mystery     37958336.06     144658074.73
          Drama     39960809.85     143552918.54
        Western     55748387.10     143540584.77
        History     53532727.27     141720573.64
         Horror     25944644.22     114205395.02
       TV Movie      5600000.00          3746.00
    Documentary             NaN 

**Insight**: Comparing average budgets and revenues by genre helps identify which genres tend to have higher production costs and generate more revenue. Zero values are excluded because they represent unknown financial data.

In [67]:

query4 = """
SELECT
    actor_name,
    COUNT(DISTINCT movie_id) AS movie_appearances
FROM cast
GROUP BY actor_name
ORDER BY movie_appearances DESC, actor_name
LIMIT 10;
"""

result4 = pd.read_sql_query(query4, connection)

print("Top 10 actors by movie appearances:")
print(result4.to_string(index=False))


Top 10 actors by movie appearances:
        actor_name  movie_appearances
 Samuel L. Jackson                 39
         Brad Pitt                 38
    Robert De Niro                 38
       Johnny Depp                 37
         Tom Hanks                 33
     Mark Wahlberg                 32
Scarlett Johansson                 32
      Willem Dafoe                 32
    Morgan Freeman                 31
        Tom Cruise                 31


**Insight**: The actors with the most movie appearances are the most frequently represented performers in this dataset. This measures appearances, not acting quality or audience popularity.

In [69]:

query5 = """
SELECT
    person_name AS director,
    COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE LOWER(TRIM(job)) = 'director'
GROUP BY person_id, person_name
HAVING COUNT(DISTINCT movie_id) > 3
ORDER BY movie_count DESC, director;
"""

result5 = pd.read_sql_query(query5, connection)

print("Directors with more than 3 movies:")
print(result5.to_string(index=False))

print("\nTotal directors found:", len(result5))

Directors with more than 3 movies:
               director  movie_count
       Steven Spielberg           27
           Ridley Scott           19
             Tim Burton           19
        Martin Scorsese           16
            Michael Bay           15
        Robert Zemeckis           15
         Clint Eastwood           14
     M. Night Shyamalan           13
       Robert Rodriguez           13
          Antoine Fuqua           12
      Christopher Nolan           11
            Danny Boyle           11
          David Fincher           11
            Guy Ritchie           11
         Hayao Miyazaki           11
              Joel Coen           11
    Paul W. S. Anderson           11
      Quentin Tarantino           11
        Roland Emmerich           11
            Zack Snyder           11
     Guillermo del Toro           10
              James Wan           10
     Jaume Collet-Serra           10
             Ron Howard           10
             Shawn Levy           10
   

**Insight**: This query identifies directors who have directed more than three movies in the dataset. It highlights prolific directors represented in this sample, rather than ranking all directors worldwide.

In [71]:
query6 = """
SELECT
    keyword_name,
    COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords
WHERE keyword_name IS NOT NULL
GROUP BY keyword_name
ORDER BY movie_count DESC, keyword_name
LIMIT 10;
"""

result6 = pd.read_sql_query(query6, connection)

print("Top 10 most frequent movie keywords:")
print(result6.to_string(index=False))

Top 10 most frequent movie keywords:
          keyword_name  movie_count
based on novel or book          449
                sequel          416
  duringcreditsstinger          296
   aftercreditsstinger          242
                murder          176
   based on true story          166
         new york city          161
               villain          160
          3d animation          155
        based on comic          154


**Insight**: The most frequent keywords reveal themes or topics that appear across many movies in the dataset. Frequent keywords describe movie metadata, not necessarily the most popular or highest-rated movies.

In [5]:
query7 = """
WITH overall_averages AS (
    SELECT
        AVG(NULLIF(CAST(budget AS REAL), 0)) AS average_budget,
        AVG(NULLIF(CAST(revenue AS REAL), 0)) AS average_revenue
    FROM movies
)
SELECT
    m.title,
    CAST(m.budget AS REAL) AS budget,
    CAST(m.revenue AS REAL) AS revenue
FROM movies AS m
CROSS JOIN overall_averages AS averages
WHERE CAST(COALESCE(m.budget, 0) AS REAL) > 0
  AND CAST(COALESCE(m.revenue, 0) AS REAL) > 0
  AND CAST(m.budget AS REAL) > averages.average_budget
  AND CAST(m.revenue AS REAL) < averages.average_revenue
ORDER BY budget DESC;
"""

result7 = pd.read_sql_query(query7, connection)

print("Movies with above-average budgets but below-average revenue:")
print(result7.to_string(index=False))

print("\nTotal movies found:", len(result7))

Movies with above-average budgets but below-average revenue:
                                                               title       budget      revenue
                                                         The Marvels 2.748000e+08 2.061368e+08
                                                             Red One 2.500000e+08 1.857008e+08
                              Wake Up Dead Man: A Knives Out Mystery 2.100000e+08 4.000000e+06
                                                   Wonder Woman 1984 2.000000e+08 1.696010e+08
                                                                Luca 2.000000e+08 5.107477e+07
                                                       Green Lantern 2.000000e+08 2.198512e+08
                                                               Mulan 2.000000e+08 6.996537e+07
                                                              Onward 2.000000e+08 1.418883e+08
                                                       Jungle Cruise 2.000000e+08 2.

**Insight**: This query identifies movies whose budgets are above the average of movies with known budgets, while their revenues are below the average of movies with known revenues. Zero values are excluded because they represent unknown data.

In [75]:
query8 = """
SELECT DISTINCT
    c.actor_name,
    m.title
FROM movies AS m
JOIN crew AS cr
    ON m.movie_id = cr.movie_id
JOIN cast AS c
    ON m.movie_id = c.movie_id
WHERE cr.person_name = 'Christopher Nolan'
  AND cr.job = 'Director'
ORDER BY m.title, c.cast_order;
"""

result8 = pd.read_sql_query(query8, connection)

print("Actors in Christopher Nolan movies:")
print(result8.head(30).to_string(index=False))

print("\nTotal actor-movie records found:", len(result8))

Actors in Christopher Nolan movies:
          actor_name         title
      Christian Bale Batman Begins
       Michael Caine Batman Begins
         Liam Neeson Batman Begins
        Katie Holmes Batman Begins
         Gary Oldman Batman Begins
      Cillian Murphy Batman Begins
       Tom Wilkinson Batman Begins
        Rutger Hauer Batman Begins
        Ken Watanabe Batman Begins
   Mark Boone Junior Batman Begins
     Fionn Whitehead       Dunkirk
           Tom Hardy       Dunkirk
        Mark Rylance       Dunkirk
     Kenneth Branagh       Dunkirk
      Cillian Murphy       Dunkirk
       Barry Keoghan       Dunkirk
        Harry Styles       Dunkirk
     Aneurin Barnard       Dunkirk
      Damien Bonnard       Dunkirk
       Lee Armstrong       Dunkirk
   Leonardo DiCaprio     Inception
Joseph Gordon-Levitt     Inception
        Ken Watanabe     Inception
           Tom Hardy     Inception
         Elliot Page     Inception
          Dileep Rao     Inception
      Cillian Murph

**Insight**: This query connects actors with movies directed by Christopher Nolan. Actors may appear in more than one film, so each actor–movie combination is a separate record.

In [77]:
query9 = """
SELECT
    g.genre_name,
    ROUND(AVG(m.vote_average), 2) AS average_rating,
    COUNT(DISTINCT m.movie_id) AS movie_count
FROM movies AS m
JOIN movie_genres AS mg
    ON m.movie_id = mg.movie_id
JOIN genres AS g
    ON mg.genre_id = g.genre_id
GROUP BY g.genre_id, g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY AVG(m.vote_average) DESC
LIMIT 1;
"""

result9 = pd.read_sql_query(query9, connection)

print("Highest-rated genre with at least 20 movies:")
print(result9.to_string(index=False))

Highest-rated genre with at least 20 movies:
genre_name  average_rating  movie_count
       War            7.36           85


**Insight**: War is the highest-rated genre among genres with at least 20 movies, averaging 7.36 across 85 movies.

In [79]:
query10 = """
SELECT
    m.title,
    COUNT(mk.keyword_id) AS keyword_count
FROM movies AS m
JOIN movie_keywords AS mk
    ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC, m.title
LIMIT 10;
"""

result10 = pd.read_sql_query(query10, connection)

print("Top 10 movies by keyword count:")
print(result10.to_string(index=False))

Top 10 movies by keyword count:
                                     title  keyword_count
                                   Smile 2             98
                                   Taken 3             70
                               Silent Hill             63
                            Twelve Monkeys             57
            War for the Planet of the Apes             54
                               The Shining             52
            Dawn of the Planet of the Apes             51
                  How to Train Your Dragon             50
 13 Hours: The Secret Soldiers of Benghazi             49
How to Train Your Dragon: The Hidden World             49


**Insight**: Smile 2 has the highest number of keywords in this dataset, with 98. Keyword counts describe the metadata available for each movie; they do not necessarily indicate movie quality or popularity.

In [82]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    connection
)

print(tables)

connection.close()

             name
0            cast
1            crew
2          genres
3    movie_genres
4  movie_keywords
5          movies


## SQL Analysis Summary

The SQLite database was successfully created and queried using Python's `sqlite3` library and Pandas.

The SQL analysis demonstrated:

- Creation and verification of six relational database tables.
- Filtering movie records using `WHERE` conditions.
- Joining related tables using primary and foreign key relationships.
- Grouping records using `GROUP BY`.
- Applying aggregate functions such as `COUNT`, `AVG` and `SUM`.
- Filtering grouped results using `HAVING`.
- Ranking results using `ORDER BY` and `LIMIT`.
- Handling unknown financial values when calculating budget and revenue averages.
- Using a Common Table Expression (CTE) to compare movie performance with overall dataset averages.
- Analysing genres, actors, directors, keywords, ratings, budgets and revenues.

These SQL queries demonstrate how the relational movie database can be used to answer meaningful analytical and business questions.